In [ ]:
include( "../args.jl" )
include( "../model.jl" )
include( "../cluster.jl" )
include( "../prob.jl" )
include( "../visual.jl" )

using Optim
using ThreadsX

home, base = expfolder( "system" );

In [ ]:
# Density and speed parameters.
include( "size-0-params.jl" )

# System size.
N = 128

# Create folder and extract critical noise.
folderdata = [findfolder( N, ρ, s; base=base, warn=false, crit=true ) for ρ ∈ ρlist, s ∈ slist]

# Extract critical noise values.
ηcdata = first.( readdlm.( folderdata.*"../order-0.5.txt" ) );

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Importing data."; dt=1.0 )

# Initialize data variables.
Cdata = Matrix{Vector{defInt}}( undef, Nρ, Ns )
Fdatalist = Matrix{Matrix{defInt}}( undef, Nρ, Ns )

# Import cluster dynamical data.
for i ∈ 1:Nρ, j ∈ 1:Ns
    Cdata[i,j] = round.( defInt, vec( readdlm( folderdata[i,j]*"cluster-frequency.txt" ) ) )
    Fdatalist[i,j] = round.( defInt, readdlm( folderdata[i,j]*"frequency-map.txt" ) )
    next!( meter )
end

In [ ]:
# Extract the clusters which can be optimized over for each parameter combination.
ιdatalist = [isnothing( Fdatalist[i,j] ) ? nothing : 0 .< vec( sum( Fdatalist[i,j]; dims=2 ) )
    for i ∈ 1:Nρ, j ∈ 1:Ns];

In [ ]:
# Initialize progress meter.
meter = Progress( Nρ*Ns, "Solving for exponents." )

# Initialize data variables.
λdata = Matrix{defFloat}( undef, Nρ, Ns )
θdatalist = Matrix{Matrix{defFloat}}( undef, Nρ, Ns )

# Solve for the parameters.
θ0 = [1.5, 1.5, 0.66]
for i ∈ 1:Nρ, j ∈ 1:Ns
    # Solve for the cluster mass distribution exponent.
    λdata[i,j] = 10.0^optimize( createzipfloss( N, Cdata[i,j] ), [log10( 1.5 )] ).minimizer[1]

    # Solve for the cluster interaction parameters.
    θdatalist[i,j] = hcat( ThreadsX.map(
        k -> ιdatalist[i,j][k] ? convert( runopt( k, Fdatalist[i,j], N, θ0 ).minimizer ) : fill( -1.0, 3 ), 1:N )... )

    # Update progress meter.
    next!( meter )
end

In [ ]:
i = 6;  j = 6

# Plot the optimal rates for changing cluster size.
plt = plot(; layout=(3,1), size=(400,600), legend=false )

for k ∈ 1:2
    scatter!( plt[k], 1:N, θdatalist[i,j][k,:]; color=:gray )
    plot!( plt[k], [1,N], fill( λdata[i,j], 2 ); color=:black, linestyle=:dash, lw=2 )
    plot!( plt[k]; xlims=(1,N), ylims=(0,3) )
end

scatter!( plt[3], 1:N, θdatalist[i,j][3,:]; color=:gray )
plot!( plt[3]; xlims=(1,N), ylims=(0,1) )

In [ ]:
if true
    # Save the cluster change parameters for each parameter case.
    for i ∈ 1:Nρ, j ∈ 1:Ns
        writedlm( folderdata[i,j]*"power-law.txt", λdata[i,j] )
        writedlm( folderdata[i,j]*"optimal-rates.txt", θdatalist[i,j] )
    end
end